In [1]:
!nvidia-smi
!nvcc --version

Tue May 19 12:02:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%writefile sum2d.cu
#include <cstdio>
#include <cuda_runtime.h>

__global__ void sum(float *d_A, int width, int height) {

    int t = width / 2;
    int s = (width % 2) ? t + 1 : t;

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < t && y < height) {
        int index1 = y * width + x;
        int index2 = y * width + x + s;

        if ((x + s) < width) {
            d_A[index1] = d_A[index1] + d_A[index2];
        }
    }
}

void print(float *A, int width, int height) {

    printf("A:\n");

    for (int y = 0; y < height; y++) {

        for (int x = 0; x < width; x++) {
            printf("%6.1f ", A[y * width + x]);
        }

        printf("\n");
    }

    printf("\n");
}

void total(int width, int height, float *A) {

    int n = width;

    printf("Starting\n");

    int size = width * height * sizeof(float);

    float *d_A;

    cudaMalloc((void**)&d_A, size);

    cudaMemcpy(d_A, A, size, cudaMemcpyHostToDevice);

    while (n > 1) {

        printf("n = %d\n", n);

        print(A, n, height);

        dim3 threadsPerBlock(16, 16);
        dim3 numBlocks((n + 15) / 16, (height + 15) / 16);

        sum<<<numBlocks, threadsPerBlock>>>(d_A, n, height);

        cudaDeviceSynchronize();

        n = (n % 2 == 0) ? n / 2 : n / 2 + 1;

        cudaMemcpy(A, d_A, size, cudaMemcpyDeviceToHost);
    }

    printf("Column sums:\n");

    for (int y = 0; y < height; y++) {
        printf("Row %d Sum = %6.1f\n", y, A[y * width]);
    }

    cudaFree(d_A);
}

int main() {

    const int height = 3;
    const int width = 10;

    float A[height][width] = {
        {2,3,3,4,4,1,6,7,4,4},
        {1,2,3,4,5,6,7,8,9,10},
        {5,5,5,5,5,5,5,5,5,5}
    };

    total(width, height, (float*)A);

    return 0;
}

Overwriting sum2d.cu


In [4]:
!nvcc sum2d.cu -o sum2d

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [5]:
!./sum2d

Starting
n = 10
A:
   2.0    3.0    3.0    4.0    4.0    1.0    6.0    7.0    4.0    4.0 
   1.0    2.0    3.0    4.0    5.0    6.0    7.0    8.0    9.0   10.0 
   5.0    5.0    5.0    5.0    5.0    5.0    5.0    5.0    5.0    5.0 

n = 5
A:
   3.0    9.0   10.0    8.0    8.0 
   1.0    6.0    7.0    4.0    4.0 
   7.0    9.0   11.0   13.0   15.0 

n = 3
A:
  11.0   17.0   10.0 
   8.0    8.0    5.0 
  10.0    7.0    4.0 

n = 2
A:
  21.0   17.0 
  10.0   13.0 
   8.0    5.0 

Column sums:
Row 0 Sum =   38.0
Row 1 Sum =   20.0
Row 2 Sum =   10.0
